Data Augmentation


In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# =========================
# PATHS
# =========================

csv_path = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train.csv"

train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train\train"

output_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

output_csv = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"

# =========================
# LOAD CSV
# =========================

df = pd.read_csv(csv_path)

# =========================
# AUGMENTATION SETTINGS
# =========================

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

# =========================
# CREATE OUTPUT FOLDERS
# =========================

os.makedirs(output_dir, exist_ok=True)

# =========================
# NEW CSV ROWS
# =========================

new_rows = []

# =========================
# PROCESS IMAGES
# =========================

for _, row in df.iterrows():

    img_id = str(row["Id"])
    label = str(row["Category"])

    # original image path
    img_path = os.path.join(
        train_dir,
        label,
        img_id + ".png"
    )

    # open image
    img = Image.open(img_path).convert("L")

    # convert to numpy
    img_array = np.array(img)

    # create class folder in augmented dataset
    class_output_dir = os.path.join(output_dir, label)
    os.makedirs(class_output_dir, exist_ok=True)

    # =========================
    # SAVE ORIGINAL IMAGE
    # =========================

    original_filename = img_id + ".png"

    original_output_path = os.path.join(
        class_output_dir,
        original_filename
    )

    img.save(original_output_path)

    # add original to csv
    new_rows.append({
        "Id": img_id,
        "Category": int(label)
    })

    # =========================
    # GENERATE AUGMENTED IMAGE
    # =========================

    img_array = img_array.reshape((1, 32, 32, 1))

    aug_iter = datagen.flow(img_array, batch_size=1)

    # number of augmentations per image
    for i in range(1):

        aug_img = next(aug_iter)[0].astype(np.uint8)

        aug_img = aug_img.reshape(32, 32)

        aug_pil = Image.fromarray(aug_img)

        aug_filename = f"{img_id}_aug{i}.png"

        aug_output_path = os.path.join(
            class_output_dir,
            aug_filename
        )

        aug_pil.save(aug_output_path)

        # add augmented image to csv
        new_rows.append({
            "Id": f"{img_id}_aug{i}",
            "Category": int(label)
        })

# =========================
# SAVE NEW CSV
# =========================

new_df = pd.DataFrame(new_rows)

new_df.to_csv(output_csv, index=False)

print("DONE")
print("Augmented dataset saved to:")
print(output_dir)

print("\nNew CSV saved to:")
print(output_csv)

print("\nTotal images:", len(new_df))

DONE
Augmented dataset saved to:
C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented

New CSV saved to:
C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv

Total images: 34000


Submission csv cleaner


In [ ]:
import pandas as pd

df = pd.read_csv('C:\\Users\\alijo\\OneDrive\\Desktop\\IVP project\\IVP-Group32\\submissionBaselineNoPreprocess.csv')

# Strip the .png extension from the Id column
df['Id'] = df['Id'].str.replace('.png', '', regex=False)

df.to_csv('C:\\Users\\alijo\\OneDrive\\Desktop\\IVP project\\IVP-Group32\\submissionBaselineNoPreprocess.csv', index=False)